In [0]:
%pip install gradio 

  Using cached gradio-6.20.0-py3-none-any.whl.metadata (17 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached huggingface_hub-1.25.1-py3-none-any.whl.metadata (16 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached tomlkit-0.14.0-py3-none-any.whl.metadata (2.8 kB)
INFO: pip is looking at multiple versions of fastapi to determine which version is compatible with other requirements. This could take a while.
  Using cached fastapi-0.141.1-py3-none-any.whl.metadata (27 kB)
  Using cached typing_inspection-0.4.2

In [0]:
%pip install gradio groq

  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-f2da3402-4670-459a-93ed-779371ba1fd2
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.14 requires confection<2.0.0,>=1.3.2, but you have confection 0.1.5 which is incompatible.
spacy 3.8.14 requires srsly<3.0.0,>=2.5.3, but you have srsly 2.5.1 which is incompatible.
spacy 3.8.14 requires weasel<2.0.0,>=1.0.0, but you have weasel 0.4.1 which is incompatible.
thinc 8.3.13 requires blis<1.4.0,>=1.3.0, but you have blis 0.7.11 which is incompatible.
thinc 8.3.13 requires confection<2.0.0,>=1.1.0, but you have confection 0.1.5 which

In [0]:
dbutils.library.restartPython()

In [0]:
import gradio as gr
import pandas as pd
import json
import requests
from pyspark.sql import SparkSession
from groq import Groq

# -------------------------------------------------------------------------
# CONFIGURACION
# -------------------------------------------------------------------------
try:
    GROQ_API_KEY = dbutils.secrets.get(scope="consultech", key="groq-key")
    SERPER_API_KEY = dbutils.secrets.get(scope="consultech", key="serper-key")
except Exception as e:
    raise ValueError(f"No se pudieron leer las claves: {str(e)}")

cliente_ai = Groq(api_key=GROQ_API_KEY)

MODELO_LLM = "llama-3.3-70b-versatile"

# -------------------------------------------------------------------------
# CARGA DE DATOS
# -------------------------------------------------------------------------
spark = SparkSession.builder.getOrCreate()
df_predicciones = spark.read.table(
    "proyecto_gestion_costos_operativos.default.predicciones_4_meses"
).toPandas()
df_predicciones['ds'] = pd.to_datetime(df_predicciones['ds'])

try:
    df_importancia_rf = spark.read.table(
        "proyecto_gestion_costos_operativos.default.importancia_variables_por_equipo"
    ).toPandas()
except Exception:
    df_importancia_rf = None

try:
    df_coeficientes_prophet = spark.read.table(
        "proyecto_gestion_costos_operativos.default.coeficientes_prophet_por_equipo"
    ).toPandas()
except Exception:
    df_coeficientes_prophet = None

try:
    df_silver = spark.read.table("proyecto_gestion_costos_operativos.default.silver_equipos").toPandas()
    matriz_correlacion_insumos = df_silver[['Price_X', 'Price_Y', 'Price_Z']].corr()
except Exception:
    matriz_correlacion_insumos = None

meses_map = {
    "Mes 1 (Sept)": 9,
    "Mes 2 (Oct)": 10,
    "Mes 3 (Nov)": 11,
    "Mes 4 (Dic)": 12
}

# -------------------------------------------------------------------------
# FUNCIONES DE CONTEXTO CORREGIDAS
# -------------------------------------------------------------------------
def obtener_contexto_datos(mes_seleccionado, equipo):
    num_mes = meses_map.get(mes_seleccionado, 9)
    # ✅ CORRECCIÓN CLAVE: mapea el texto visible al valor guardado en la tabla
    eq_bd = 'Price_Equipo1' if equipo == 'Equipo 1' else 'Price_Equipo2'
    filtro_mes = df_predicciones[
        (df_predicciones['ds'].dt.month == num_mes) &
        (df_predicciones['Equipo'] == eq_bd)
    ]
    
    if filtro_mes.empty:
        return f"No hay datos de pronostico para {equipo} en {mes_seleccionado}."

    filtro_ordenado = filtro_mes.sort_values('yhat', ascending=False)
    fila_max = filtro_ordenado.iloc[0]
    fecha_max = fila_max['ds'].strftime('%Y-%m-%d')
    valor_max = fila_max['yhat']
    minimo = filtro_mes['yhat'].min()
    prom = filtro_mes['yhat'].mean()
    ancho = (filtro_mes['yhat_upper'] - filtro_mes['yhat_lower']).mean()

    texto = "DATOS PRINCIPALES PARA " + equipo + " - " + mes_seleccionado + ":\n"
    texto += "- Fecha con costo mas alto: " + fecha_max + " - $" + f"{valor_max:,.2f}" + "\n"
    texto += "- Costo maximo del mes: $" + f"{valor_max:,.2f}" + "\n"
    texto += "- Costo minimo: $" + f"{minimo:,.2f}" + "\n"
    texto += "- Promedio mensual: $" + f"{prom:,.2f}" + "\n"
    texto += "- Rango de incertidumbre promedio: $" + f"{ancho:,.2f}" + "\n\n"
    texto += "Detalle diario:\n"
    for _, row in filtro_ordenado.iterrows():
        fecha = row['ds'].strftime('%Y-%m-%d')
        texto += "- " + fecha + ": $" + f"{row['yhat']:,.2f}" + " (90%: $" + f"{row['yhat_lower']:,.2f}" + " - $" + f"{row['yhat_upper']:,.2f}" + ")\n"

    return texto

def obtener_contexto_explicativo(equipo):
    col = 'Equipo_1' if equipo == 'Equipo 1' else 'Equipo_2'
    texto = "Factores que explican el costo de " + equipo + ":\n\n"
    if df_importancia_rf is not None and col in df_importancia_rf.columns:
        rf = df_importancia_rf[['Variable', col]].sort_values(col, ascending=False)
        texto += "1. Importancia segun RandomForest:\n"
        for _, r in rf.iterrows():
            texto += "   " + r['Variable'] + ": " + f"{r[col]:.4f}" + "\n"
    else:
        texto += "1. Sin datos de RandomForest (ejecuta Notebook 2)\n"
    col_abs = f"{col}_abs"
    if df_coeficientes_prophet is not None and col_abs in df_coeficientes_prophet.columns:
        pr = df_coeficientes_prophet[['regressor', col, col_abs]].sort_values(col_abs, ascending=False)
        texto += "\n2. Impacto segun Prophet:\n"
        for _, r in pr.iterrows():
            texto += "   " + r['regressor'] + ": coef=" + f"{r[col]:.4f}" + " | peso=" + f"{r[col_abs]:.4f}" + "\n"
    else:
        texto += "\n2. Sin datos de Prophet (ejecuta Notebook 3)\n"
    if matriz_correlacion_insumos is not None:
        texto += "\n3. Correlacion entre insumos:\n" + matriz_correlacion_insumos.round(2).to_string()
    return texto

# -------------------------------------------------------------------------
# BUSQUEDA EXTERNA
# -------------------------------------------------------------------------
def buscar_noticias_mercado(query: str):
    if not SERPER_API_KEY:
        return "Busqueda no configurada. Se usan solo datos internos."
    try:
        resp = requests.post(
            "https://google.serper.dev/news",
            headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
            json={"q": query, "gl": "co", "hl": "es", "num": 5},
            timeout=10
        )
        resp.raise_for_status()
        noticias = resp.json().get("news", [])
        if not noticias:
            return f"No hay resultados recientes para: '{query}'."
        res = "Informacion de mercado: " + query + "\n"
        for n in noticias[:5]:
            res += "- " + n['title'] + " (" + n.get('source','fuente') + "): " + n.get('snippet','') + " | " + n.get('link','') + "\n"
        return res
    except Exception as e:
        return f"Error en busqueda: {str(e)}. Se usan solo datos del proyecto."

herramientas = [
    {
        "type": "function",
        "function": {
            "name": "buscar_noticias_mercado",
            "description": "Busca precios de insumos, inflacion, logistica y noticias del sector construccion. No inventes informacion.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Ej: precio acero Colombia 2026"}
                },
                "required": ["query"]
            }
        }
    }
]

# -------------------------------------------------------------------------
# MOTOR DEL AGENTE
# -------------------------------------------------------------------------
def motor_agente_ia_real(mensaje, historial, mes_actual, equipo_actual):
    ctx_datos = obtener_contexto_datos(mes_actual, equipo_actual)
    ctx_explica = obtener_contexto_explicativo(equipo_actual)
    system_prompt = (
        "Eres analista experto en costos de construccion y gestion de costos operativos. Responde solo con evidencia: "
        f"Pronosticos: {ctx_datos}. Factores explicativos: {ctx_explica}. "
        "REGLAS: 1. Resume cifras, no listes todo salvo que te pidan detalle. "
        "2. Para mercado usa la herramienta de busqueda; NUNCA inventes datos. "
        "3. Si falla la busqueda, avisa claramente. "
        "4. Para 'por que': si RF y Prophet coinciden -> hallazgo robusto; si no -> explica lineal vs no lineal y correlacion entre insumos. "
        "5. Usa lenguaje claro y estructurado."
    )

    mensajes_limpios = []
    for msg in historial:
        mensajes_limpios.append({
            "role": msg["role"],
            "content": msg["content"]
        })

    mensajes_api = [{"role": "system", "content": system_prompt}] + mensajes_limpios + [{"role": "user", "content": mensaje}]

    try:
        resp = cliente_ai.chat.completions.create(
            model=MODELO_LLM,
            messages=mensajes_api,
            tools=herramientas,
            tool_choice="auto",
            temperature=0.4,
            max_tokens=700
        )
        msg = resp.choices[0].message

        if msg.tool_calls:
            mensajes_api.append(msg.model_dump(exclude_none=True))
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                res_busq = buscar_noticias_mercado(args.get("query", ""))
                mensajes_api.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": res_busq
                })
                resp_final = cliente_ai.chat.completions.create(
                    model=MODELO_LLM,
                    messages=mensajes_api,
                    temperature=0.4,
                    max_tokens=700
                )
                respuesta = resp_final.choices[0].message.content
        else:
            respuesta = msg.content

    except Exception as e:
        respuesta = f"Error: {str(e)}"

    return "", historial + [{"role": "user", "content": mensaje}, {"role": "assistant", "content": respuesta}]

# -------------------------------------------------------------------------
# INTERFAZ CORREGIDA
# -------------------------------------------------------------------------
with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("# Agente IA - ConsulTech Analytics")
    mes_state = gr.State()
    eq_state = gr.State()
    with gr.Column() as p_meses:
        gr.Markdown("### 1. Selecciona mes")
        sel_mes = gr.Radio(list(meses_map.keys()), label="Periodo")
        btn1 = gr.Button("Siguiente", variant="primary")
    with gr.Column(visible=False) as p_equipos:
        gr.Markdown("### 2. Selecciona equipo")
        sel_eq = gr.Radio(["Equipo 1", "Equipo 2"], label="Equipos")
        with gr.Row():
            v_mes = gr.Button("Volver", variant="secondary")
            btn2 = gr.Button("Analizar", variant="primary")
    with gr.Column(visible=False) as p_chat:
        gr.Markdown("### 3. Conversa con el agente")
        info = gr.Markdown("")
        chat = gr.Chatbot()
        entrada = gr.Textbox(label="Pregunta sobre costos, riesgos o factores explicativos:")
        with gr.Row():
            enviar = gr.Button("Enviar", variant="primary")
            salir = gr.Button("Cambiar mes", variant="stop")

    def ir_equipos(m):
        return gr.update(visible=False), gr.update(visible=True), m
    def ir_meses():
        return gr.update(visible=True), gr.update(visible=False)
    def iniciar(m, eq):
        if not eq:
            return gr.update(visible=True), gr.update(visible=False), eq, "", []
        txt = "Agente listo: " + eq + " | " + m + "\nPregunta por pronosticos, explicaciones o riesgos de mercado."
        # ✅ Limpia el chat al cambiar selección para no mezclar datos viejos
        return gr.update(visible=False), gr.update(visible=True), eq, txt, []
    def reiniciar():
        return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), None, None, []

    btn1.click(ir_equipos, sel_mes, [p_meses, p_equipos, mes_state])
    v_mes.click(ir_meses, outputs=[p_meses, p_equipos])
    btn2.click(iniciar, [mes_state, sel_eq], [p_equipos, p_chat, eq_state, info, chat])
    salir.click(reiniciar, outputs=[p_meses, p_equipos, mes_state, eq_state, chat])
    enviar.click(motor_agente_ia_real, [entrada, chat, mes_state, eq_state], [entrada, chat])
    entrada.submit(motor_agente_ia_real, [entrada, chat, mes_state, eq_state], [entrada, chat])

demo.launch(share=True)

/root/.ipykernel/3379/command-7999569091466485-2451556659:220: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Base()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://60481fba4c6c4b17d4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
